df=pd.read_csv("https://raw.githubusercontent.com/awais-DS/Data/refs/heads/main/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [2]:
import numpy as np
import pandas as pd


In [3]:
df=pd.read_csv("https://raw.githubusercontent.com/awais-DS/Data/refs/heads/main/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [4]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [6]:
df=df[["gender","tenure","Contract","Dependents","MonthlyCharges","Churn"]]

In [7]:
df.columns

Index(['gender', 'tenure', 'Contract', 'Dependents', 'MonthlyCharges',
       'Churn'],
      dtype='object')

In [8]:
df.duplicated().sum()

np.int64(159)

In [9]:
df=df.drop_duplicates().copy()

In [10]:
df.duplicated().sum()

np.int64(0)

In [11]:
df.columns

Index(['gender', 'tenure', 'Contract', 'Dependents', 'MonthlyCharges',
       'Churn'],
      dtype='object')

In [12]:
df["gender"].head()

,gender
0,Female
1,Male
2,Male
3,Male
4,Female


In [13]:
# Selecting x column
x=df.drop(columns=["Churn"])

In [14]:
# Selecting y column
y=df["Churn"]

In [15]:
x.shape

(6884, 5)

In [16]:
y.shape

(6884,)

In [17]:
# Importing library for spliting
from sklearn.model_selection import train_test_split

In [18]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=0)

In [19]:
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(5507, 5)
(5507,)
(1377, 5)
(1377,)


In [20]:
x_train.head()

,gender,tenure,Contract,Dependents,MonthlyCharges
1736,Female,1,Month-to-month,No,19.50
1660,Male,47,Month-to-month,No,25.40
1430,Female,23,One year,No,24.80
2823,Male,39,Two year,No,20.45
5042,Male,19,One year,Yes,19.80


In [21]:
df["Contract"].unique()

array(['Month-to-month', 'One year', 'Two year'], dtype=object)

# Feature Engineering

In [22]:
# Importing Encoder from scikit learn
from sklearn.preprocessing import OneHotEncoder,OrdinalEncoder,LabelEncoder


In [23]:
# Initailizing Encoders accodint to Data Types
gender_enc=OneHotEncoder(drop="first",sparse_output=False) # Nomianal Encoding
dependent_enc=OneHotEncoder(drop="first",sparse_output=False) # Nomianal Encoding

contract_enc=OrdinalEncoder(categories=[["Month-to-month","One year","Two year"]]) # Ordinal 

Churn_enc=LabelEncoder()

In [24]:
# Enoding Training Data (fit_transform)
x_train["gender"]=gender_enc.fit_transform(x_train[["gender"]]).ravel()
x_train["Dependents"]=dependent_enc.fit_transform(x_train[["Dependents"]]).ravel()

x_train["Contract"]=contract_enc.fit_transform(x_train[["Contract"]]).ravel()

In [25]:
y_train=Churn_enc.fit_transform(y_train)

In [26]:
print(y_train[:5])

[1 0 0 0 0]


In [27]:
x_train.head()

,gender,tenure,Contract,Dependents,MonthlyCharges
1736,0.0,1,0.0,0.0,19.50
1660,1.0,47,0.0,0.0,25.40
1430,0.0,23,1.0,0.0,24.80
2823,1.0,39,2.0,0.0,20.45
5042,1.0,19,1.0,1.0,19.80


In [28]:
# Encoding Testing Data (transform only)
x_test["gender"]=gender_enc.transform(x_test[["gender"]]).ravel()
x_test["Dependents"]=dependent_enc.transform(x_test[["Dependents"]]).ravel()

x_test["Contract"]=contract_enc.transform(x_test[["Contract"]]).ravel()

y_test=Churn_enc.transform(y_test)

# Creating Pipelines

In [30]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

In [ ]:
preprocessor=ColumnTransformer(
    transformers=[
        ("gender_step",OneHotEncoder(drop="first",sparse_output=False),
        )
    ]
)

In [31]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline


preprocessor = ColumnTransformer(
    transformers=[
        # Nominal / Binary encoding (returns 0 or 1)
        ('gender_step', OneHotEncoder(drop='first', sparse_output=False), ['gender']),
        ('dependent_step', OneHotEncoder(drop='first', sparse_output=False), ['Dependents']),
        
        # Ordinal encoding (maps strictly to 0, 1, 2)
        ('contract_step', OrdinalEncoder(categories=[['month-to-month', 'One year', 'Two year']]), ['Contract'])
    ],
    remainder='passthrough' # Keeps other features (like tenure) safe
)



data_cleaning_pipeline = Pipeline(steps=[
    ('encoder_prep', preprocessor)
])

print("Data-only Pipeline successfully created!")


Data-only Pipeline successfully created!


In [32]:
#Merging my Data
clean_df=pd.DataFrame(x_train)
clean_df["Churn"]=y_train

In [33]:
clean_df.head()

,gender,tenure,Contract,Dependents,MonthlyCharges,Churn
1736,0.0,1,0.0,0.0,19.50,1
1660,1.0,47,0.0,0.0,25.40,0
1430,0.0,23,1.0,0.0,24.80,0
2823,1.0,39,2.0,0.0,20.45,0
5042,1.0,19,1.0,1.0,19.80,0


In [38]:
import pandas as pd

# 1. Combine your numeric x_train features and y_train labels back together
clean_df = pd.DataFrame(x_train)
clean_df['Churn'] = y_train

# 2. Save it directly to the local folder (No drive prefix needed!)
clean_df.to_csv('clean_data.csv', index=False)
print("Data successfully saved locally in your project folder!")


Data successfully saved locally in your project folder!


In [39]:
import pandas as pd

# Load the file to see if your numbers look correct
test_load = pd.read_csv('clean_data.csv')
print("File exists! Data shape is:", test_load.shape)
print(test_load.head(2))


File exists! Data shape is: (5507, 6)
   gender  tenure  Contract  Dependents  MonthlyCharges  Churn
0     0.0       1       0.0         0.0            19.5      1
1     1.0      47       0.0         0.0            25.4      0
